# Faster R-CNN（PyTorch 版）

用 `torchvision` 加载预训练 Faster R-CNN，跑推理，拆解内部结构。

**目标：** 理解两阶段检测器 —— RPN 提候选 → ROI Head 分类 + 精修。

```
输入图像
  → Backbone (ResNet-50 + FPN)  → 多尺度特征图
  → RPN                          → ~1000 个 proposal
  → ROI Align + Detection Head   → 最终检测框 + 类别
```


In [ ]:
import torch
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.transforms import functional as FT
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Running on: {device}')


## 1. 加载预训练模型

`fasterrcnn_resnet50_fpn` — ResNet-50 骨干 + FPN 颈 + RPN + ROI Head。

`pretrained=True` 自动下载 COCO 预训练权重（91 类，含背景）。


In [ ]:
# 加载预训练 Faster R-CNN
model = fasterrcnn_resnet50_fpn(pretrained=True)
model.to(device)
model.eval()

print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

# COCO 类别名（不包含 0 = 背景）
COCO_CLASSES = [
    '__background__', 'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus',
    'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'N/A', 'stop sign',
    'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow',
    'elephant', 'bear', 'zebra', 'giraffe', 'N/A', 'backpack', 'umbrella', 'N/A',
    'N/A', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball',
    'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket',
    'bottle', 'N/A', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl',
    'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza',
    'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'N/A', 'dining table',
    'N/A', 'N/A', 'toilet', 'N/A', 'tv', 'laptop', 'mouse', 'remote', 'keyboard',
    'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'N/A',
    'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush'
]
print(f'COCO classes: {len(COCO_CLASSES)} (including background)')


## 2. 加载测试图像

项目目录下的 `catdog.jpg`（猫 + 狗，正好试检测）。


In [ ]:
# 读取图像
img = Image.open('catdog.jpg').convert('RGB')
print(f'Image size: {img.size}')

fig, ax = plt.subplots(figsize=(6, 4))
ax.imshow(img)
ax.set_title('Input Image')
ax.axis('off')
plt.show()


## 3. 推理

Faster R-CNN 的输入是一个 **Tensor 列表**（每张图一个 Tensor），输出是一个字典列表：

| 输出 key | 含义 | 形状 |
|---|---|---|
| `boxes` | 检测框 (x1, y1, x2, y2) | (N_det, 4) |
| `labels` | 类别标签 | (N_det,) |
| `scores` | 置信度 | (N_det,) |


In [ ]:
# 预处理：PIL -> Tensor [0,1], 加 batch 维
x = FT.to_tensor(img).unsqueeze(0).to(device)
print(f'Input tensor shape: {x.shape}')  # (1, 3, H, W)

# 推理
with torch.no_grad():
    predictions = model(x)

pred = predictions[0]  # 第一张图的预测结果
print(f'\nDetected {len(pred["boxes"])} objects')
print(f'\n--- Raw predictions ---')
for i in range(min(len(pred['boxes']), 10)):
    box = pred['boxes'][i].cpu().numpy()
    label = COCO_CLASSES[pred['labels'][i].item()]
    score = pred['scores'][i].item()
    print(f'  [{label:15s}] score={score:.3f}  box=({box[0]:.0f}, {box[1]:.0f}, {box[2]:.0f}, {box[3]:.0f})')


## 4. 可视化检测结果

设定置信度阈值 `score_thresh=0.7`，过滤低质量检测。


In [ ]:
def draw_boxes(img, pred, score_thresh=0.7):
    """在 PIL Image 上画检测框。"""
    draw_img = img.copy()
    draw = ImageDraw.Draw(draw_img)

    colors = ['#00FF00','#FF4444','#4488FF','#FFAA00','#AA44FF',
              '#00CCCC','#FF69B4','#888800','#FF8800','#44FF44']

    for i in range(len(pred['boxes'])):
        score = pred['scores'][i].item()
        if score < score_thresh:
            continue

        box = pred['boxes'][i].cpu().numpy()
        label = COCO_CLASSES[pred['labels'][i].item()]
        color = colors[pred['labels'][i].item() % len(colors)]

        draw.rectangle([(box[0], box[1]), (box[2], box[3])],
                       outline=color, width=3)
        draw.text((box[0], box[1] - 12), f'{label} {score:.2f}',
                  fill=color)

    return draw_img


draw_img = draw_boxes(img, pred, score_thresh=0.7)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.imshow(img); ax1.set_title('Original'); ax1.axis('off')
ax2.imshow(draw_img); ax2.set_title('Faster R-CNN Detections (score > 0.7)'); ax2.axis('off')
plt.tight_layout()
plt.show()


## 5. 拆解：Backbone, RPN, ROI Head 各输出什么

`torchvision` 的 Faster R-CNN 内部结构：

```
model.backbone(x)          → feature_maps (dict of P2-P6)
model.rpn(images, features)→ proposals (List[Tensor])
model.roi_heads(features, proposals) → detections
```


In [ ]:
# ---- 1. Backbone 特征提取 ----
model.eval()
with torch.no_grad():
    features = model.backbone(x)

print('=== Backbone output (FPN levels) ===')
for level, feat in features.items():
    print(f'  {level}: shape = {tuple(feat.shape)}')
print(f'  P2-P5: high-res features for detection')
print(f'  P6: extra coarse level for large objects')

# ---- 2. RPN 生成 proposals ----
with torch.no_grad():
    # RPN 内部会算 objectness + bbox delta, 然后 decode + NMS
    images = [FT.to_tensor(t) for t in [img]]
    images = [t.to(device) for t in images]
    proposals, proposal_losses = model.rpn(images, features, None)

print(f'\n=== RPN output ===')
print(f'  Proposals per image: {[p.shape[0] for p in proposals]}')
print(f'  Shape of first 5: {proposals[0][:5].shape if len(proposals[0]) >= 5 else "N/A"}')
print(f'  Training losses: {proposal_losses}  # None during eval')
print(f'\n  RPN does: feature -> 3x3 conv -> two 1x1 convs ->')
print(f'           -> objectness (fg/bg) + bbox_delta for each anchor')
print(f'           -> decode + NMS -> ~1000 proposals per image')

# ---- 3. ROI Head 分类 + 精修 ----
with torch.no_grad():
    detections, detector_losses = model.roi_heads(features, proposals, None)

print(f'\n=== ROI Head output ===')
dets = detections[0]
for key in dets.keys():
    print(f'  {key}: shape = {dets[key].shape}')
print(f'\n  ROI Head does:')
print(f'    ROI Align each proposal -> 7x7xC feature')
print(f'    -> FC layers -> two branches:')
print(f'      - classification: K+1 classes (including bg)')
print(f'      - bbox regression: 4*(K+1) offsets per proposal')
print(f'    -> NMS per class -> final detections')


## 6. 不同置信度阈值对比

降低阈值会出更多框（包括低质量检测）。


In [ ]:
thresholds = [0.9, 0.7, 0.5, 0.3]
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for ax, thresh in zip(axes.flat, thresholds):
    draw_img = draw_boxes(img, pred, score_thresh=thresh)
    ax.imshow(draw_img)
    count = (pred['scores'] > thresh).sum().item()
    ax.set_title(f'score > {thresh}  ({count} detections)')
    ax.axis('off')

plt.suptitle('Faster R-CNN: Different Score Thresholds', y=1.01, fontsize=14)
plt.tight_layout()
plt.show()


## 7. 再看一张图：单独猫

`cat1.jpg` — 项目中已有的猫图。


In [ ]:
try:
    img_cat = Image.open('cat1.jpg').convert('RGB')
    print(f'Image size: {img_cat.size}')

    x_cat = FT.to_tensor(img_cat).unsqueeze(0).to(device)

    with torch.no_grad():
        pred_cat = model(x_cat)[0]

    draw_cat = draw_boxes(img_cat, pred_cat, score_thresh=0.5)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
    ax1.imshow(img_cat); ax1.set_title('Original'); ax1.axis('off')
    ax2.imshow(draw_cat); ax2.set_title('Faster R-CNN (score > 0.5)'); ax2.axis('off')
    plt.tight_layout()
    plt.show()

    for i in range(min(len(pred_cat['boxes']), 5)):
        label = COCO_CLASSES[pred_cat['labels'][i].item()]
        score = pred_cat['scores'][i].item()
        box = pred_cat['boxes'][i].cpu().numpy()
        print(f'  [{label:15s}] score={score:.3f}  box=({box[0]:.0f},{box[1]:.0f},{box[2]:.0f},{box[3]:.0f})')

except FileNotFoundError:
    print('cat1.jpg not found, skip.')


## 小结

### Faster R-CNN 的三个阶段

| 阶段 | 组件 | 输入 | 输出 |
|---|---|---|---|
| 1. 特征提取 | ResNet-50 + FPN | 图像 (3, H, W) | 5 层多尺度特征图 (P2-P6) |
| 2. 区域提议 | RPN | 特征图 | ~1000 个 proposal (坐标 + objectness) |
| 3. 分类精修 | ROI Align + Head | 特征图 + proposals | N 个检测 (class + refined box + score) |

### 和 SSD / YOLO 的核心区别

| | Faster R-CNN | SSD / YOLO |
|---|---|---|
| 检测方式 | 先提候选再分类 | 一步到位 |
| 小物体 | 好（ROI Align 精修） | 较差（浅层语义弱） |
| 速度 | 慢 (~0.2s) | 快 (实时) |
| mAP | 高 | 略低 |

### 延伸

- 把骨干换成 ResNet-101 / MobileNet
- 换 `MaskRCNN` 加分割分支（API 完全一样）
- Fine-tune 自己的数据集（改 `num_classes`）
